# English → Darija: data preparation
Run the cells in order. Steps: load → inspect → clean → EDA → half-sample → split → tokenizer.

## 0. Install

In [ ]:
%pip install -q datasets sentencepiece sacrebleu

## 1. Imports and config

In [ ]:
import re, os, random, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42          # same seed for the whole team
FRACTION = 1    # share of the data we keep
os.makedirs("data", exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)

## 2. Load the dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("imomayiz/darija-english", "sentences")
df = ds["sentences"].to_pandas()

# Alternative if you downloaded the file by hand:
# df = pd.read_csv("sentences.csv")

print(df.shape)
print(df.columns.tolist())
df.sample(15, random_state=0)

## 3. Set the column names
Look at the output above, then edit these two lines to match the real column names.

In [ ]:
EN_COL = "eng"
DAR_COL = "darija"   # or "darija" for the Latin version

## 4. Clean

In [ ]:
# If the dataframe was already renamed in memory, reload it from ds
if "en" in df.columns and "dar" in df.columns:
    df = ds["sentences"].to_pandas()

df = df[[EN_COL, DAR_COL]].rename(columns={EN_COL: "en", DAR_COL: "dar"})
n0 = len(df)

df = df.dropna()
for c in ["en", "dar"]:
    df[c] = df[c].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df = df[(df["en"] != "") & (df["dar"] != "")]
n1 = len(df)

df = df.drop_duplicates().reset_index(drop=True)
n2 = len(df)

same = df["en"].str.lower() == df["dar"].str.lower()
df = df[~same].reset_index(drop=True)
n3 = len(df)

print(f"start: {n0} | after empty/NaN: {n1} | after duplicates: {n2} | after identical pairs: {n3}")

## 5. Script detection (Arabic vs Latin) and lengths

In [ ]:
AR = re.compile(r"[\u0600-\u06FF]")
LAT = re.compile(r"[A-Za-z]")

def script_of(s):
    a, l = len(AR.findall(s)), len(LAT.findall(s))
    if a + l == 0:
        return "other"
    return "arabic" if a >= l else "latin"

df["script"] = df["dar"].map(script_of)
df["en_len"] = df["en"].str.split().str.len()
df["dar_len"] = df["dar"].str.split().str.len()

print(df["script"].value_counts())
print(df["script"].value_counts(normalize=True).round(3))
df.sample(10, random_state=1)

## 6. Take a random half (stratified by script)

In [ ]:
df_half = (
    df.groupby("script", group_keys=False)
      .sample(frac=FRACTION, random_state=SEED)
      .reset_index(drop=True)
)
print(len(df), "->", len(df_half))

## 7. Check that the half looks like the full data

In [ ]:
rows = []
for name, d in [("full", df), ("half", df_half)]:
    rows.append({
        "set": name,
        "pairs": len(d),
        "en_len_mean": d["en_len"].mean(),
        "en_len_median": d["en_len"].median(),
        "dar_len_mean": d["dar_len"].mean(),
        "dar_len_median": d["dar_len"].median(),
        "arabic_share": (d["script"] == "arabic").mean(),
        "latin_share": (d["script"] == "latin").mean(),
    })
pd.DataFrame(rows).round(3)

## 8. Exploratory data analysis (for the report)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(df_half["en_len"], bins=40); ax[0].set_title("English length (words)")
ax[1].hist(df_half["dar_len"], bins=40); ax[1].set_title("Darija length (words)")
plt.tight_layout(); plt.savefig("data/eda_lengths.png", dpi=150); plt.show()

def vocab_stats(series, name):
    cnt = collections.Counter(w.lower() for s in series for w in s.split())
    once = sum(1 for c in cnt.values() if c == 1)
    print(f"{name}: vocab={len(cnt)}, words seen once={once} ({once/len(cnt):.1%})")
    print("  top 15:", cnt.most_common(15))

vocab_stats(df_half["en"], "English")
vocab_stats(df_half["dar"], "Darija")
print(df_half[["en_len", "dar_len"]].describe().round(2))

## 9. Train / validation / test split (80 / 10 / 10)

In [ ]:
key = df_half["en"].str.lower()
g = np.random.RandomState(SEED).permutation(key.unique())
k = len(g) // 10
part = {s: "val" for s in g[:k]} | {s: "test" for s in g[k:2*k]}
split = key.map(lambda s: part.get(s, "train"))
train, val, test = (df_half[split == s] for s in ["train", "val", "test"])

for name, d in [("train", train), ("val", val), ("test", test)]:
    d.reset_index(drop=True).to_csv(f"data/{name}.csv", index=False)
    print(name, len(d))

## 10. Train the tokenizers (on the TRAIN set only)
If you get a vocab-size error, lower `VOCAB` to 4000.

In [ ]:
import sentencepiece as spm

VOCAB = 3000

def write_lines(series, path):
    with open(path, "w", encoding="utf-8") as f:
        for s in series:
            f.write(s.replace("\n", " ") + "\n")

write_lines(train["en"], "data/train.en.txt")
write_lines(train["dar"], "data/train.dar.txt")

# split_by_number=False: in Arabizi the digits 3/7/9/2/5/8 are LETTERS, so the default (True) cut every
# word at its digits (3la -> _3/la, ch7al -> _ch/7/al) and no vocab piece could mix digits and letters.
# EN is lowercased (nmt_nfkc_cf); DAR keeps "identity" because S/D/T capitals are emphatic consonants.
def train_spm(txt, prefix, norm):
    spm.SentencePieceTrainer.train(
        input=txt, model_prefix=prefix, vocab_size=VOCAB,
        model_type="bpe", character_coverage=1.0,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        normalization_rule_name=norm, hard_vocab_limit=False,
        split_by_number=False,
    )

train_spm("data/train.en.txt", "data/spm_en", "nmt_nfkc_cf")
train_spm("data/train.dar.txt", "data/spm_dar", "identity")

## 11. Check the tokenizers and choose MAX_LEN

In [ ]:
import sentencepiece as spm
import numpy as np

sp_en = spm.SentencePieceProcessor(model_file="data/spm_en.model")
sp_dar = spm.SentencePieceProcessor(model_file="data/spm_dar.model")

for _, r in train.sample(3, random_state=0).iterrows():
    print("EN :", r["en"])
    print("   ", sp_en.encode(r["en"], out_type=str))
    print("DAR:", r["dar"])
    print("   ", sp_dar.encode(r["dar"], out_type=str), "\n")

en_lens = [len(sp_en.encode(s)) for s in train["en"]]
dar_lens = [len(sp_dar.encode(s)) for s in train["dar"]]
for name, l in [("EN", en_lens), ("DAR", dar_lens)]:
    print(name, "tokens  mean/95%/99%/max:",
          round(np.mean(l), 1), int(np.percentile(l, 95)), int(np.percentile(l, 99)), max(l))

MAX_LEN = int(max(np.percentile(en_lens, 99), np.percentile(dar_lens, 99))) + 2  # +2 for <bos>/<eos>
print("MAX_LEN =", MAX_LEN)

**Note (fix):** the tokenizers used for training are now produced by `retrain_tokenizers.py` with exactly the settings above (`split_by_number=False`, EN `nmt_nfkc_cf`, DAR `identity`, BPE 3000). It also prints the checks of this section (segmentations, unk rate, MAX_LEN). The original tokenizers are kept in `data/tok_orig/` for the ablation.

## Done
The splits are rebuilt by `scripts/prepare_data.py` (same logic, sha256-checked) and the tokenizers by `scripts/retrain_tokenizers.py`. The CSVs are not redistributed in the repository (dataset licence): run `make data` instead. Next: the LSTM cells and the encoder-decoder (`src/darija_nmt/`).